# The Baseline MemAgent

----------

### One agent, one store, and the twenty decisions its memory loop makes

An agent's memory is a loop. 

It **retains** what it reads, **refines** what it kept, **recalls** what a turn needs and **reuses** what worked. 

Inside that loop sit many small judgements: does a new topic start here, is this worth keeping, which store should be searched, which tool should run.

Most systems answer those judgements with a threshold, a rule, or a full language-model prompt that writes prose nobody reads. 

A **decision model** is a different tool for the same job: it reads a state, takes typed questions and returns calibrated probabilities, with no generated text.

### Agent memory in one page

Agent memory is how an agent retains, refines, recalls and reuses what it learns, across turns, sessions and agents. Without it, every call replays the whole history and corrections die with the session. With it, the context stays roughly flat and corrections become memory. Three principles frame everything after this: memory exists to make an agent more reliable, believable and capable; forget, don't delete; and the four Rs (retain, refine, recall, reuse) are the loop every memory system runs.

<img src="img/reference/p02_agent_memory.png" alt="What agent memory is: where it lives (parametric, working, persistent) and the taxonomy of short-term and long-term memory." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 2. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p05_without_and_with_memory.png" alt="Side by side: the loop without memory replays everything, about 17.5k tokens by turn ten; with memory it stays about 3.5k a turn." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 5. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p06_principles.png" alt="Three principles of agent memory: the RBCs, forget don't delete, and the four Rs." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 6. The full PDF is in <code>docs/</code>.</sub>


This notebook builds the agent and lists the decisions inside its memory loop. Notebooks 01 to 07 then take one technique each, show how it is built today and where it breaks, swap the judgement inside it for a decision model, and measure both on the same labelled data. Nothing here uses a decision model yet. [MemoRizz](https://github.com/RichmondAlake/memorizz) manages all of the memory, and every later swap is a small change against this agent.

| Notebook | Decision points | The swap: today → with a decision model |
|---|---|---|
| **00 baseline** | all twenty, listed | nothing yet |
| 01 semantic chunking | 1 | a similarity drop → "does a new topic start here?" |
| 02 reranking | 16 | on topic → "does this chunk answer the query?" |
| 03 semantic cache | the cache in front of recall | a cosine threshold → "can this cached answer serve this query?" |
| 04 memory routing | 13, 14 | the nearest store → "which memory types does this need?" |
| 05 consolidation and dedup | 6, 7, 8, 10 | cosine above 0.95 → same, refines, contradicts, complements or unrelated |
| 06 memory extraction | 2, 3 | always run → "anything durable, and what type?" first |
| 07 tool selection | 19 | top-k by similarity → "which tool, if any?" |
| 08 scoreboard | all seven | the seven scorecards side by side |
| 09 three decision models | all seven | d1, Jev and OpenAI Decisions on the same questions |

**Live path.** ⭐ marks the sections worth showing. Every cell has saved output, so you can talk through it without re-running.

| Section | What to show | Minutes |
|---|---|---|
| ⭐ 1.1 | The twenty decisions, grouped by R | 3 |
| ⭐ 2.4 | Three turns on the running agent | 2 |
| ⭐ 3.2 | What recall pulled for one turn | 2 |

### Reference architecture

Every type of memory gets a place to live, and every arrow is a data flow: the write path after each turn, the read path on every turn, and background jobs on a schedule.

<img src="img/reference/p08_reference_architecture.png" alt="The agent memory reference architecture: user, agent harness, context window and LLM; the memory manager's write path, read path and background jobs; and the memory database with six stores." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 8. The full PDF is in <code>docs/</code>.</sub>

The pieces these notebooks use, as built here:

Every component the notebooks use. The store is a folder of files with a FAISS index, so everything runs on a laptop with no database.

<img src="img/00_reference_architecture.png" alt="Reference architecture: a user turn reaches a MemAgent, which talks to the chat LLM and a MemoRizz store and passes twenty memory decisions to either a typical path or a decision model." width="1184">

<details><summary>Mermaid source</summary>

```mermaid
flowchart TB
    U([User turn]):::user --> A[MemAgent]:::app
    A <--> L[Chat LLM<br/>gpt-4o-mini]:::llm
    A --> D{{"Twenty memory decisions<br/>Retain · Refine · Recall · Reuse"}}:::gate
    D --> T[Typical path<br/>thresholds · rules · LLM prompts]:::code
    D --> M[Decision model<br/>LiquidAI d1-2B, notebooks 01 to 07]:::dm
    A <--> S[(MemoRizz store<br/>local FAISS files)]:::db
    S --- K[Knowledge base]:::mem
    S --- C[Conversation memory]:::mem
    S --- B[Toolbox]:::tool
    classDef user fill:#1e1b4b,stroke:#a78bfa,color:#ede9fe
    classDef app fill:#0c2a4d,stroke:#60a5fa,color:#dbeafe
    classDef llm fill:#3b1d0a,stroke:#fb923c,color:#ffedd5
    classDef dm fill:#052e2b,stroke:#2dd4bf,color:#ccfbf1
    classDef db fill:#3b0a1e,stroke:#f472b6,color:#fce7f3
    classDef mem fill:#1f2a07,stroke:#a3e635,color:#ecfccb
    classDef tool fill:#2e1065,stroke:#c084fc,color:#f3e8ff
    classDef gate fill:#3f2a00,stroke:#fbbf24,color:#fef3c7
    classDef code fill:#111827,stroke:#94a3b8,color:#e5e7eb
```

</details>

# Part 1 · The Twenty Decisions

## 1.1 · ⭐ The inventory

> 💡 **Key Term — Decision point:** a place in the memory loop where the system picks between known outcomes. The answer is a yes or no, one option from a list, or a level on a scale, never free text.

The harness keeps the inventory as data, so every notebook can import it. Each row names the question, how it is usually implemented, the decision-model primitive that fits it, and where it is built.

Every box on the read and write paths hides a decision, marked D:

<img src="img/reference/p16_every_d_is_a_decision.png" alt="Every D is a decision: the reference architecture with each decision point marked." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 16. The full PDF is in <code>docs/</code>.</sub>

In [1]:
import sys
sys.path.insert(0, "..")

import harness
from harness import measure

print("keys set:", harness.load_env())
measure.use_notebook("00_baseline")

keys set: ['OPENAI_API_KEY', 'TYPESAFE_API_KEY', 'HF_TOKEN']


In [2]:
import pandas as pd
from harness.decision_points import inventory

pd.set_option("display.max_colwidth", None, "display.max_rows", None)
table = inventory()
table.set_index(["r", "n"])[["name", "typical", "d1_primitive", "notebook"]]

name  \
r      n                                 
Retain 1                Chunk boundary   
       2                Retention gate   
       3                   Memory type   
       4              Sensitivity gate   
       5            Extraction quality   
       6               Duplicate check   
       7            Conflict detection   
       8                    Resolution   
Refine 9                     Staleness   
       10  Consolidation and promotion   
       11               Memory utility   
       12              Trace selection   
Recall 13                  Recall gate   
       14               Memory routing   
       15           Query rewrite need   
       16                    Reranking   
       17            Sufficiency check   
       18             Injection screen   
Reuse  19     Tool and skill selection   
       20              Tool-call guard   

                                                                                        typical  \
r      n                                                                                          
Retain 1    cosine drop between neighbouring sentences, cut at a percentile (or a fixed window)   
       2                      none: extraction runs on every turn (or an LLM importance prompt)   
       3                                                                             LLM prompt   
       4                                                                        regex, then LLM   
       5                                                                           LLM as judge   
       6   cosine threshold (0.95 is common; memorizz treats summaries as duplicates from 0.97)   
       7                                                                 LLM prompt, or nothing   
       8                                                                             LLM prompt   
Refine 9                                                  age decay (memorizz RetentionPlanner)   
       10                                                          LLM (memorizz consolidation)   
       11                                                                         access counts   
       12                                                                                   LLM   
Recall 13                                                                       always retrieve   
       14                                           nearest store description, or an LLM prompt   
       15                                                 LLM (RetrievalPolicy.query_expansion)   
       16                                                      cosine order, or a cross-encoder   
       17                                                                                   LLM   
       18                                                                     LLM or classifier   
Reuse  19                                                   embedding top-k, then the LLM picks   
       20                                                                       allowlist rules   

                                                            d1_primitive  \
r      n                                                                   
Retain 1                                                    Noul per gap   
       2                                  Noul, a gate before extraction   
       3                      Choice: preference, fact, procedure, event   
       4                      regex mask, then Choice on the local model   
       5                                                            Noul   
       6                    Choice of five relations, per candidate pair   
       7                                    the same Choice: contradicts   
       8   the same Choice picks skip, update, supersede, link or insert   
Refine 9                                                            Noul   
       10                             the same Choice, run on a schedule   
       11         

**What to watch:** the `typical` column. Retain and Recall lean on LLM prompts, Refine leans on rules such as age decay and access counts, and only some rows are already cheap. Those rows are the case for swapping, and also the reason to measure instead of assuming.

## 1.2 · The twenty decisions, stage by stage

The inventory above, drawn as the loop. **Retain** is the write path after each turn, **Refine** runs in the background, **Recall** is the read path on every turn, and **Reuse** is the agent acting. Each page lists how every decision is made today and the decision-model primitive that would fit it.

<img src="img/reference/p21_retain.png" alt="Retain, the write path after the turn: eight decisions from chunk boundary to resolution." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 21. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p22_refine.png" alt="Refine, background jobs over the store: staleness, consolidation, utility and trace selection." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 22. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p23_recall.png" alt="Recall, the read path on every turn: recall gate, routing, query rewrite, reranking, sufficiency and the injection screen." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 23. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p24_reuse.png" alt="Reuse, procedural memory into action: tool and skill selection, and the tool-call guard." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 24. The full PDF is in <code>docs/</code>.</sub>

## 1.3 · What a decision model is

> 💡 **Key Term — Decision model:** a model that takes a **state** (what is being judged), a **question**, and the **answers it is allowed to give**, and returns a probability for each answer in one forward pass. It writes no text, so code can act on the result directly: threshold it, route on it, or pass the unsure cases to an LLM.

It sits between a classifier and an LLM: the speed and bounded output of a classifier, with the question supplied at request time like a prompt. It answers three question shapes: a **Noul** (yes or no, returning P(yes)), a **Choice** (one of several named options) and a **Score** (a level on an ordered scale).

<img src="img/reference/p17_what_a_decision_model_is.png" alt="What a decision model is: state, question and allowed answers in; one forward pass; a probability per answer out." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 17. The full PDF is in <code>docs/</code>.</sub>

<img src="img/reference/p18_classifier_llm_decision_model.png" alt="Classifier, LLM and decision model compared: what you give, what comes back, how to ask a new question, confidence, latency and cost, and where each fits." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 18. The full PDF is in <code>docs/</code>.</sub>

## 1.4 · How these notebooks call it

Every notebook asks the decision model through `harness/d1.py`, a thin wrapper written for these notebooks around the model's own code.

| Piece | Who wrote it | What it does |
|---|---|---|
| `d1.noul`, `d1.choice`, `d1.score` | these notebooks | build a question as a plain dict, in Liquid's request format |
| `d1.decide(state, questions)`, `d1.decide_batch(items)` | these notebooks | send one request, or many, and return the answers |
| `model.system_one`, `model.system_one_batch` | **Liquid**, shipped with the model | read the state once and answer every question in one forward pass |

On this laptop the two lines below return the same answers: the wrapper calls Liquid's method.

```python
d1.decide(state, questions)
d1.model().system_one(state, questions)["answers"]
```

What the wrapper adds:
- **One call for any model.** `D1_BACKEND=jev` (or `openai`, or Liquid's `hosted`), or `with d1.using("jev"):` for one block, sends the same questions elsewhere without changing a notebook.
- **Batching that suits a laptop.** Groups of 32 through the local model, with a smaller token budget; eight hosted requests at a time.
- **Hosted plumbing.** Retries on rate limits, server errors and timeouts, and OpenAI's different request shape translated both ways, so every answer comes back in d1's shape.
- **Counting.** `d1.usage` tallies requests, input tokens and refusals, which is how each notebook prices its decisions.

## 1.5 · What gets built


Seven swaps are built end to end, one notebook each: semantic chunking, reranking, the semantic cache, memory routing, consolidation and deduplication, memory extraction and tool selection. The semantic cache sits in front of recall rather than being one of the twenty; the other six cover eleven of the twenty decision points. The rest stay listed with their primitive, so the inventory is complete without trying to do everything.

With a decision model in place, every D in the architecture can go to one model in one pass:

<img src="img/reference/p25_architecture_with_a_decision_model.png" alt="The same reference architecture, with a decision model answering every decision." width="872">

<sub>From <i>The Decision on Decision Models</i>, page 25. The full PDF is in <code>docs/</code>.</sub>

In [3]:
listed = table["notebook"].eq("listed only")
print(table.assign(built=~listed).groupby("r", observed=True)["built"].agg(["sum", "size"]))
print(f"\n{(~listed).sum()} of {len(table)} decision points have a notebook")

        sum  size
r                
Retain    6     8
Refine    1     4
Recall    3     6
Reuse     1     2

11 of 20 decision points have a notebook


> 📌 **Key Takeaways — Part 1**
> - The memory loop is a list of typed questions, and writing them down is the first useful step.
> - Each one already has an answer in most systems. The comparison is against that answer, not against nothing.

# Part 2 · Build the Agent

## 2.1 · A store and a knowledge base

> 🧠 **Memory type — semantic (knowledge base):** durable reference text, split into chunks and embedded so a query can find the right passage.

The input in every notebook is the same synthetic support call: about 1,500 words across eight topics. `ingest_knowledge` chunks, embeds and stores it. `chunking_strategy="fixed"` is the first decision point, answered by a character count.

In [4]:
import shutil
from memorizz import FileSystemConfig, FileSystemProvider, KnowledgeBase, MemAgentBuilder, RetrievalPolicy, Toolbox
from memorizz.enums import MemoryType

ROOT = harness.STORE / "baseline"
shutil.rmtree(ROOT, ignore_errors=True)
provider = FileSystemProvider(FileSystemConfig(root_path=ROOT))

In [5]:
text, gold = harness.demo()
print(f"{len(text.split())} words, {len(gold['topics'])} topics")

kb = KnowledgeBase(memory_provider=provider)
kb_id = kb.ingest_knowledge(text, namespace="demo", chunking_strategy="fixed",
                            chunk_size=1000, chunk_overlap=100)
kb.last_ingest

1490 words, 8 topics


{'knowledge_base_id': '18570e8d-2bca-44a7-8582-69b621de13c9',
 'chunks': 9,
 'embedded': 9}

## 2.2 · Eleven demo tools

> 🧠 **Memory type — procedural (toolbox):** what the agent can do, stored and embedded like any other memory so tools can be found by meaning.

`Toolbox` stores each function with its signature and an embedding of its docstring. Eleven tools is enough for an embedding search to rank the wrong one first sometimes, which notebook 07 measures.

In [6]:
import textwrap
from harness import demo_tools

demo_tools.bind(provider)
toolbox = Toolbox(memory_provider=provider)
for tool in demo_tools.TOOLS:
    toolbox.register_tool(tool)
print(textwrap.fill(", ".join(tool.__name__ for tool in demo_tools.TOOLS), 100))

search_memory, get_weather, convert_currency, create_calendar_event, summarise_thread, lookup_order,
reset_password, recommend_book, send_email, post_to_slack, generate_report


## 2.3 · Build the agent

`RetrievalPolicy` says how much recall does on each turn: fetch up to 20 candidates, keep 5. The agent runs on `gpt-4o-mini`. Three settings matter for this text:

- `scoring=harness.RELEVANCE_ONLY` ranks recall on relevance alone. MemoRizz blends recency, importance and relevance, and min-max normalises each across the candidates. Every chunk here was written in the same second, so recency carries no signal, yet min-max stretches those millisecond differences across the whole 0 to 1 range and the first chunks written score lowest. Keeping recency on drops the chunk that answers the second turn below.
- `max_chars_per_item=None` keeps chunks whole. The default of 600 characters would cut a 1,000-character chunk, and the chunker has already decided the size.
- `.with_automations_enabled(False)` stops "Remind me..." being taken as a request to schedule a reminder, which is not what this notebook is about.

In [7]:
def builder():
    return (MemAgentBuilder()
        .with_instruction("You are the LLMCamp demo assistant. Use memory before you answer.")
        .with_llm_config({"provider": "openai", "model": harness.CHAT_MODEL})
        .with_memory_provider(provider)
        .with_knowledge_base(kb_id)
        .with_toolbox(toolbox)
        .with_automations_enabled(False))

POLICY = dict(candidate_limit=20, max_items=5, max_chars_per_item=None, scoring=harness.RELEVANCE_ONLY)
agent = builder().with_retrieval_policy(RetrievalPolicy(**POLICY)).build()

## 2.4 · ⭐ Three turns

Each turn is timed and priced, so even the baseline leaves rows in `results/`. Notebook 03 uses this cost of a full turn to price a cache miss.

**What to watch:** the third turn says "them". The agent has to resolve that from the conversation and pick the calendar tool on its own. The exact wording of the replies can change between runs.

In [8]:
TURNS = ["What language does the customer prefer?",
         "Remind me when the migration deadline is.",
         "Book a 30 minute call with them next Tuesday."]

for turn in TURNS:
    with measure.timed("agent_turn"):
        reply = agent.run(turn, memory_id="demo")
    tokens = agent.get_last_run_usage() or {}
    measure.cost("agent_turn", tokens.get("input_tokens", 0), tokens.get("output_tokens", 0))
    print(f"> {turn}\n{textwrap.fill(reply, 100)}\n")

> What language does the customer prefer?
The customer, Priya Nair, prefers Python as their programming language.

> Remind me when the migration deadline is.
The migration deadline is Friday, the 14th of November. After that date, the old platform will go
into read-only mode.

> Book a 30 minute call with them next Tuesday.
I've booked a 30-minute call with Priya Nair for next Tuesday at 15:00 UTC.



> 📌 **Key Takeaways — Part 2**
> - An agent with memory is a store, a knowledge base, a toolbox and a retrieval policy.
> - Every later notebook replaces one judgement inside this agent and leaves the rest alone.

# Part 3 · What MemoRizz Did

Memory is easy to treat as magic, so this part reads back what actually happened.

## 3.1 · The written turns

> 🧠 **Memory type — episodic (conversation):** every turn is kept, in order, with its role and timestamp.

In [9]:
rows = provider.retrieve_conversation_history_ordered_by_timestamp("demo")
pd.DataFrame([{"role": r["role"], "content": str(r["content"])[:80],
               "importance": r.get("importance"), "source": r.get("importance_source")} for r in rows])

,role,content,importance,source
0,user,What language does the customer prefer?,0.65,heuristic
1,assistant,"The customer, Priya Nair, prefers Python as their programming language.",0.55,heuristic
2,user,Remind me when the migration deadline is.,0.65,heuristic
3,assistant,"The migration deadline is Friday, the 14th of November. After that date, the old",0.55,heuristic
4,user,Book a 30 minute call with them next Tuesday.,0.50,heuristic
5,assistant,I've booked a 30-minute call with Priya Nair for next Tuesday at 15:00 UTC.,0.40,heuristic


**What to watch:** the `importance` and `source` columns. That is decision point 2 being answered on every write, by a rule. Notebook 06 puts a decision model in front of what gets written.

## 3.2 · ⭐ What recall pulled

`retrieve_by_query` is the search recall uses. The scores are cosine similarities, and everything below `limit` is a candidate the agent never considered.

In [10]:
hits = provider.retrieve_by_query(TURNS[0], memory_store_type=MemoryType.KNOWLEDGE_BASE,
                                  limit=20, namespace="demo")
pd.DataFrame([{"chunk": r["chunk_index"], "score": round(r["score"], 3),
               "starts with": r["content"][:70].replace("\n", " ")} for r in hits])

,chunk,score,starts with
0,1,0.496,"nsion request has to reach your account manager by the 7th, so I will"
1,2,0.488,and asyncio? The SDK ships an async client. Priya: Yes to both. We st
2,4,0.390,ents team asked me to collect lunch choices while I had you. Priya: Th
3,0,0.384,"Sam: Thanks for calling the Harbourline support line, this is Sam. Wha"
4,6,0.366,"hat did they find? Priya: We were charged for 60 seats, but we only ha"
5,3,0.328,newal on the load balancer. The status page estimates a fix within two
6,8,0.321,"y password expired, and the reset email never comes. Sam: I can trigge"
7,7,0.307,mmend to a new manager? Sam: I would start with Thinking in Systems by
8,5,0.279,The one at ten o'clock with Elena? Priya: That's the one. Marcus is tr


**What to watch:** the fixed window cut the transcript wherever 1,000 characters ran out, so the answer about Python shares a chunk with whatever came next. That is why notebook 01 exists.

## 3.3 · The tool pre-filter

`get_most_similar_tools` is decision point 19's typical first step: an embedding search over tool descriptions.

In [11]:
for turn in TURNS:
    found = toolbox.get_most_similar_tools(turn, limit=3)
    print(f"{turn}\n   {[(t['name'], round(t['score'], 3)) for t in found]}")

What language does the customer prefer?
   [('post_to_slack', 0.336), ('convert_currency', 0.322), ('recommend_book', 0.281)]
Remind me when the migration deadline is.
   [('search_memory', 0.315), ('reset_password', 0.22), ('create_calendar_event', 0.218)]
Book a 30 minute call with them next Tuesday.
   [('create_calendar_event', 0.408), ('recommend_book', 0.309), ('generate_report', 0.254)]


> 📌 **Key Takeaways — Part 3**
> - The store shows every decision the loop made, so each one can be inspected and measured.
> - Similarity scores rank candidates, but they do not say whether a candidate helps.

# Part 4 · Turn On the Typical Judgements

The agent above uses free rules. MemoRizz can also answer two decisions with the language model, and these are the baselines the later notebooks have to beat.

## 4.1 · Routing and importance by LLM

- **Decision 3, memory type:** `.with_memory_routing(router="llm", min_confidence=0.5)`.
- **Decision 2, importance:** the rater reads `MEMORIZZ_IMPORTANCE_RATER`, overlaid by `RetrievalPolicy(scoring={"importance": {...}})`. Mode `llm` rates each row on the background model, off the user path.

In [12]:
from memorizz.llms.llm_factory import create_llm_provider

background = create_llm_provider({"provider": "openai", "model": harness.CHAT_MODEL})
judging = RetrievalPolicy(**{**POLICY, "scoring": {**harness.RELEVANCE_ONLY, "importance": {"mode": "llm"}}})
judged_agent = (builder().with_background_model(background)
                .with_memory_routing(router="llm", min_confidence=0.5)
                .with_retrieval_policy(judging).build())
judged_agent.memory_routing.router, judged_agent._importance_rater().config.mode

('llm', 'llm')

## 4.2 · One turn with the judgements on

The turn states a fact worth keeping. `close()` waits for the background work, so the store is complete before it is read.

In [13]:
turn = "Priya's director is Hannah Cole, and she signs off every extension request."
with measure.timed("judged_turn"):
    print(textwrap.fill(judged_agent.run(turn, memory_id="judged"), 100))
judged_agent.close()

I've stored that Hannah Cole is Priya's director, and she signs off on all extension requests. If
you need any more information or have further updates, feel free to share!


{'ok': True,
 'cleanup': None,
 'closed': {'memory_upkeep': True,
  'embedding_backfill': True,
  'memory_provider': True},
 'errors': []}

In [14]:
written = provider.retrieve_conversation_history_ordered_by_timestamp("judged")
print([(r["role"], r.get("importance"), r.get("importance_source")) for r in written])

found = provider.retrieve_by_query("Hannah Cole", memory_store_type=MemoryType.ENTITY_MEMORY, limit=2)
print([(row["name"], row["entity_type"]) for row in found])

[('user', 0.9, 'llm'), ('assistant', 0.8, 'llm')]


[('Hannah Cole', 'person')]


**What to watch:** `importance_source` is now `llm`, and the router filed the fact about a named person into entity memory. Both answers cost extra model calls on a turn the user never sees. Notebook 06 asks first whether a turn is worth extracting at all.

> 📌 **Key Takeaways — Part 4**
> - Two typical judgements are one line each in MemoRizz, and each is a language-model call.
> - They run after the turn, so nobody notices the latency, but the bill still arrives.

# Part 5 · Measure

## 5.1 · One measuring tool

`harness/measure.py` gives every notebook the same two calls. `timed(label)` records wall-clock seconds, and `cost(label, input_tokens, output_tokens)` records tokens and dollars from a price table. Both append to `results/<notebook>.jsonl`, and notebook 08 reads the seven scorecards from those files and nothing else.

In [15]:
print("USD per million tokens (input, output):")
for model, rates in measure.PRICES.items():
    print(f"  {model:26} {rates}")

USD per million tokens (input, output):
  d1.1                       (0.02, 0.0)
  jev                        (0.042, 0.0)
  gpt-6-luna                 (0.1, 0.0)
  gpt-4o-mini                (0.15, 0.6)
  text-embedding-3-small     (0.02, 0.0)
  local                      (0.0, 0.0)


In [16]:
results = pd.read_json(measure.path(), lines=True)
results.groupby(["kind", "label"]).agg(rows=("label", "size"), seconds=("seconds", "mean"),
                                       usd=("usd", "sum"))

rows   seconds       usd
kind label                                
cost agent_turn      3       NaN  0.003841
time agent_turn      3  2.011275  0.000000
     judged_turn     1  6.910118  0.000000

> 📌 **Key Takeaways — Part 5**
> - One timing call and one cost call are enough to compare any two implementations.
> - Everything from here on writes to `results/`, so the final table is computed, not remembered.